# BrainScan-v2 — Étape 1 : Classification des tumeurs cérébrales (GoogLeNet)

## 1.3 — Construction du modèle GoogLeNet pré-entraîné et choix des hyperparamètres

**Objectifs du notebook**

1. Comprendre pourquoi une **implémentation Keras de GoogLeNet** est nécessaire
   (Keras 3 a retiré `keras.applications.googlenet`).
2. Construire le backbone et **transférer les poids ImageNet** pré-entraînés.
3. **Vérifier numériquement** que l'implémentation Keras est équivalente à
   `torchvision.models.googlenet` (le contrôle le plus important du notebook).
4. **Adapter la partie de classification** (couches fully connected) à nos 4 classes.
5. Fixer les **hyperparamètres** : learning rate, fonction de perte, optimiseur.

In [ ]:
# Cellule 0 — Imports, configuration et chemins du projet
"""Cellules communes réutilisées par les 4 notebooks de l'étape 1 (classification)."""

import json
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import matplotlib

if matplotlib.get_backend().lower() == "agg":  # exécution headless (Docker / nbconvert)
    try:
        matplotlib.use("module://matplotlib_inline.backend_inline")
    except Exception:
        pass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from tensorflow import keras

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)

# --------------------------------------------------------------------------------------
# Configuration du projet
# --------------------------------------------------------------------------------------
def _detect_project_root() -> Path:
    """Racine du projet (compatible exécution locale et exécution dans Docker /app)."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data" / "Data").is_dir() and (candidate / "src").is_dir():
            return candidate
    if Path("/app/data/Data").is_dir():
        return Path("/app")
    raise FileNotFoundError("Dataset introuvable : « data/Data » manquant depuis " + str(Path.cwd()))


PROJECT_ROOT = _detect_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "Data"
PROCESSED_DIR = DATA_DIR / "processed" / "classification"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = ARTIFACTS_DIR / "reports"

for _d in (ARTIFACTS_DIR, MODELS_DIR, REPORTS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

#: Graine aléatoire globale (reproductibilité de bout en bout).
SEED = 42

#: Extensions d'images autorisées par le cahier des charges.
ALLOWED_EXTENSIONS = {".jpeg", ".jpg", ".bmp", ".png"}

#: Libellés lisibles pour l'affichage (correspondance dossier -> pathologie).
CLASS_DISPLAY = {
    "glioma": "Gliome",
    "meningioma": "Méningiome",
    "notumor": "Aucune tumeur",
    "pituitary": "Hypophyse",
}

#: Classes du dataset, déduites des noms de dossiers (ordre alphabétique).
class_names = sorted(p.name for p in RAW_DIR.iterdir() if p.is_dir())

print(f"PROJECT_ROOT    : {PROJECT_ROOT}")
print(f"RAW_DIR         : {RAW_DIR}")
print(f"PROCESSED_DIR   : {PROCESSED_DIR}")
print(f"ARTIFACTS_DIR   : {ARTIFACTS_DIR}")
print(f"MODELS_DIR      : {MODELS_DIR}")
print(f"Classes         : {class_names}")


### 1.18 — Pourquoi une implémentation Keras de GoogLeNet ?

| Source | État | Verdict |
|---|---|---|
| `keras.applications.googlenet` | supprimé depuis Keras 2.13 | ❌ indisponible |
| `tf_keras.applications.googlenet` | absent de la 2.21 | ❌ indisponible |
| poids Keras historiques (`.h5`) | retirés du bucket GCS | ❌ indisponible |
| ONNX Model Zoo + `onnx2keras` | conversion **approximative** (écart relatif ~20 % sur les logits) | ⚠️ peu fiable |
| **`torchvision.models.googlenet`** | poids ImageNet officiels `IMAGENET1K_V1` | ✅ **retenu** |

La solution retenue : **reconstruire l'architecture GoogLeNet en Keras 3** et y transférer
les poids `torchvision` (mapping explicite couche par couche). L'implémentation est dans
`src/brainscan/training/googlenet.py` et réutilisable par le pipeline Airflow et l'API.

In [ ]:
# Cellule 3.1 — Import du module BrainScan
from brainscan.training.googlenet import (
    GOOGLENET_CLASSES,
    IMAGENET_MEAN,
    IMAGENET_STD,
    build_googlenet,
    build_input_pipeline,
    set_reproducible,
)

set_reproducible(SEED)

print(f"Module           : {PROJECT_ROOT / 'src' / 'brainscan' / 'training' / 'googlenet.py'}")
print(f"Statistiques ImageNet : mean={IMAGENET_MEAN} std={IMAGENET_STD}")
print(f"Classes du projet    : {GOOGLENET_CLASSES}")


### 1.19 — Construction du backbone et transfert des poids ImageNet

`build_googlenet()` :

1. construit la stem (`conv1` 7×7/2, `maxpool1`, `conv2`, `conv3`, `maxpool2`),
2. enchaîne les **9 blocs Inception** avec les pools intercalaires (`maxpool3`, `maxpool4`),
3. applique le **Global Average Pooling** (1024 caractéristiques),
4. remplace la tête d'origine par un classifieur **fully connected** pour nos 4 classes,
5. **transfère les poids ImageNet** de torchvision (57 couches convolution/BatchNorm).

In [ ]:
# Cellule 3.2 — Construction du modèle
NUM_CLASSES = len(class_names)
IMAGE_SIZE = 224

model = build_googlenet(
    num_classes=NUM_CLASSES,
    class_names=class_names,
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
    image_size=IMAGE_SIZE,
    weights="imagenet",          # transfert des poids pré-entraînés
    freeze_backbone=True,        # phase 1 : extraction de caractéristiques
    dense_units=128,
    dropout=0.5,
    include_preprocessing=True,  # Resizing + Rescaling + Normalization intégrés
    augment=True,                # flip / flou / bruit — actifs en training uniquement
)

model.class_names = class_names
print(f"Modèle construit : {model.name}")
print(f"Entrée  : {model.input_shape}")
print(f"Sortie : {model.output_shape}  -> {class_names}")


### 1.20 — Contrôle numérique : équivalence Keras ↔ torchvision

C'est le point critique du notebook. On compare, **couche par couche**, les activations
du modèle Keras (poids transférés) et celles du modèle torchvision original, sur les
mêmes entrées normalisées ImageNet.

Un écart de l'ordre de `1e-5` (précision `float32`) confirme que l'implémentation Keras
est **numériquement fidèle** : les poids pré-entraînés sont donc correctement
transférés, et le backbone est un vrai GoogLeNet ImageNet.

In [ ]:
# Cellule 3.3 — Vérification numérique (torchvision vs Keras)
import torch
from torchvision.models import googlenet

torch.manual_seed(SEED)

tv_model = googlenet(
    weights="IMAGENET1K_V1",
    transform_input=False,   # le modèle attend une entrée déjà normalisée ImageNet
    aux_logits=True,
    init_weights=False,
).eval()

# Captures des activations intermédiaires de torchvision
tv_feats: dict[str, np.ndarray] = {}


def _hook(name):
    def _fn(_module, _inp, out):
        tv_feats[name] = out.detach().numpy()
    return _fn


for name, module in [
    ("conv1", tv_model.conv1),
    ("maxpool1", tv_model.maxpool1),
    ("conv3", tv_model.conv3),
    ("maxpool2", tv_model.maxpool2),
    ("inception3a", tv_model.inception3a),
    ("inception3b", tv_model.inception3b),
    ("maxpool3", tv_model.maxpool3),
    ("inception4e", tv_model.inception4e),
    ("maxpool4", tv_model.maxpool4),
    ("inception5b", tv_model.inception5b),
    ("gap", tv_model.avgpool),
]:
    module.register_forward_hook(_hook(name))

# Modèle Keras *sans* prétraitement pour comparer à torch directement
keras_eval_model = build_googlenet(
    num_classes=NUM_CLASSES,
    class_names=class_names,
    weights="imagenet",
    include_preprocessing=False,
    freeze_backbone=True,
)

layer_by_name = {layer.name: layer for layer in keras_eval_model.layers}

rng = np.random.default_rng(SEED)
sample = rng.normal(size=(4, IMAGE_SIZE, IMAGE_SIZE, 3)).astype("float32")
sample_imagenet = ((sample - np.array(IMAGENET_MEAN) * 255.0) / (np.array(IMAGENET_STD) * 255.0)).astype("float32")

with torch.no_grad():
    tv_model(torch.from_numpy(sample_imagenet.transpose(0, 3, 1, 2)))

CHECKPOINTS = [
    ("conv1", "conv1_relu"),
    ("maxpool1", "maxpool1"),
    ("conv3", "conv3_relu"),
    ("maxpool2", "maxpool2"),
    ("inception3a", "inception3a_concat"),
    ("inception3b", "inception3b_concat"),
    ("maxpool3", "maxpool3"),
    ("inception4e", "inception4e_concat"),
    ("maxpool4", "maxpool4"),
    ("inception5b", "inception5b_concat"),
    ("gap", "gap"),
]

rows = []
for tv_name, keras_name in CHECKPOINTS:
    sub = keras.Model(keras_eval_model.input, layer_by_name[keras_name].output)
    k_out = sub.predict(sample_imagenet, verbose=0)
    t_out = tv_feats[tv_name]
    if t_out.ndim == 4:
        t_out = t_out.transpose(0, 2, 3, 1)
    t_out = t_out.reshape(k_out.shape)
    err = float(np.abs(t_out - k_out).max())
    rows.append({"couche": tv_name, "shape": str(k_out.shape), "ecart_max_abs": err,
                 "ecart_relatif": err / (float(np.abs(t_out).max()) + 1e-9)})

equiv_df = pd.DataFrame(rows)
print("Écart maximal entre torchvision et Keras, couche par couche :\n")
print(equiv_df.to_string(index=False, float_format=lambda v: f"{v:.3e}"))

max_rel = float(equiv_df["ecart_relatif"].max())
print(f"\nÉcart relatif maximal : {max_rel:.3e}")
assert max_rel < 1e-3, "Le transfert de poids n'est pas fidèle !"
print("-> Transfert de poids FIDÈLE : le backbone Keras est un GoogLeNet ImageNet exact.")


### 1.21 — Stratégie de gel des couches

Transfer learning en **deux temps** :

| Phase | Backbone | Learning rate | Objectif |
|---|---|---|---|
| **A** — *feature extraction* | gelé | `1e-3` | apprendre le classifieur sur des characteristics ImageNet génériques |
| **B** — *fine-tuning* | dégelé (derniers blocs) | `1e-5` | adapter les features aux IRM cérébrales |

On commence par la phase A : elle converge vite et fournit une base de comparaison
(« baseline »), puis la phase B affine le résultat.

In [ ]:
# Cellule 3.4 — Comptage des paramètres entraînables / gelés
trainable = sum(int(np.prod(w.shape)) for w in model.trainable_weights)
frozen = sum(int(np.prod(w.shape)) for w in model.non_trainable_weights)

print(f"Paramètres entraînables : {trainable:>12,}")
print(f"Paramètres gelés        : {frozen:>12,}")
print(f"Total                   : {trainable + frozen:>12,}")
print(f"\nPart entraînable : {trainable / (trainable + frozen):.1%}")

print("\nCouches du classifieur (n Newly ajoutées) :")
for layer in model.layers:
    if layer.name.startswith("classifier"):
        n = int(sum(np.prod(w.shape) for w in layer.get_weights()))
        print(f"  {layer.name:<22s} {type(layer).__name__:<22s} {n:>10,} paramètres")


### 1.22 — Hyperparamètres d'entraînement

| Hyperparamètre | Valeur | Justification |
|---|---|---|
| **Optimiseur** | `Adam` | convergence rapide, adaptation automatique du pas d'apprentissage, standard pour le transfer learning |
| **Learning rate** | `1e-3` (phase A) / `1e-5` (phase B) | `1e-3` pour la tête aléatoirement initialisée ; `1e-5` pour le fine-tuning (évite de détruire les features ImageNet) |
| **Loss** | `categorical_crossentropy` | labels one-hot fournis par `image_dataset_from_directory(label_mode="categorical")` |
| **Batch size** | `32` | compromis mémoire/vitesse sur CPU ; gradient suffisamment stable |
| **Epochs** | `15` (A) + `8` (B) | avec `EarlyStopping(patience=3)` et `ReduceLROnPlateau` |
| **Metrics** | `accuracy`, `precision`, `recall`, `f1` | évaluation multi-classe demandée |

La décroissance du learning rate est gérée par `ReduceLROnPlateau` (facteur `0.5`,
patience `2`).

In [ ]:
# Cellule 3.5 — Compilation du modèle
LR_PHASE_A = 1e-3
LR_PHASE_B = 1e-5
BATCH_SIZE = 32
EPOCHS_A = 15
EPOCHS_B = 8

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR_PHASE_A),
    loss="categorical_crossentropy",
    metrics=[
        keras.metrics.CategoricalAccuracy(name="accuracy"),
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
        keras.metrics.F1Score(name="f1", average="macro"),
    ],
)

model.summary(line_length=110)


### 1.23 — Architecture détaillée (extrait)

Vue condensée du modèle : pipeline d'entrée (prétraitement + augmentation), backbone
GoogLeNet, puis le classifieur fully connected.

In [ ]:
# Cellule 3.6 — Architecture détaillée
model.summary(expand_nested=True, line_length=110)


### 1.24 — Contrôle de bon fonctionnement avant entraînement

Prédiction sur un lot du jeu d'entraînement : le modèle doit produire une distribution
de probabilités valide (somme = 1) sur les 4 classes.

In [ ]:
# Cellule 3.7 — Prédiction "à blanc" (avant entraînement)
train_ds_preview = tf.keras.utils.image_dataset_from_directory(
    str(PROCESSED_DIR / "train"),
    labels="inferred",
    label_mode="categorical",
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=8,
    shuffle=True,
    seed=SEED,
)

xb_preview, yb_preview = next(iter(train_ds_preview))
probs = model.predict(xb_preview, verbose=0)

print(f"Forme des probabilités : {probs.shape}")
print(f"Somme des probabilités par exemple : {probs.sum(axis=1).round(6)}")
print(f"\nClasse prédite (avant entraînement) pour les {len(probs)} premiers exemples :")
print([class_names[int(p.argmax())] for p in probs])
print(f"\nVraies classes : {[class_names[int(y.argmax())] for y in yb_preview]}")
print("\n-> Les prédictions initiales sont (quasi) aléatoires : l'entraînement va les corriger.")


In [ ]:
# Cellule 3.8 — Sauvegarde du modèle non entraîné (pour inspection)
MODEL_DIR = MODELS_DIR / "classification"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

INIT_MODEL_PATH = ARTIFACTS_DIR / "googlenet_untrained.keras"
model.save(INIT_MODEL_PATH)
print(f"Modèle non entraîné sauvegardé : {INIT_MODEL_PATH}")
print(f"Taille : {INIT_MODEL_PATH.stat().st_size / 1e6:.1f} Mo")


### 1.25 — Récapitulatif

Le modèle est prêt. Le **notebook 4** réalise l'entraînement (phases A et B), l'évaluation
(accuracy / precision / recall / F1), la matrice de confusion, la sauvegarde du modèle
entraîné et le suivi MLflow.